# QC Neural Benchmark

Evaluates the 1D CNN architecture (`QCNeuralNet`) on the hybrid→paired transfer task.

**Hypothesis**: Per-recording z-scoring (`RecordingNormalizer`) + shift-invariant CNNs
should outperform LightGBM trained on hybrid and evaluated on paired units (baseline R²≈−0.3).

**Setup**:
- Train on hybrid recordings (same as LightGBM baseline)
- Evaluate on paired matched units
- Compare: dummy | lightgbm | neural

**Requirements**: `pip install torch` (CPU is fine for this dataset size)

In [1]:
# Install dependencies (uncomment as needed)
# !pip install torch --index-url https://download.pytorch.org/whl/cpu
# !pip install -e '../' lightgbm

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable when running locally
repo_root = Path(".").resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from qc_framework import (
    QCDataLoader, QCDataCleaner,
    FeatureSetBuilder,
)
from qc_neural import RecordingNormalizer, NeuralTrainer

In [3]:
# ---------------------------------------------------------------------------
# Configuration — adjust PARQUET_PATH to your local artifact
# ---------------------------------------------------------------------------
PARQUET_PATH = "../data/raw/33000_ROWS.parquet"   # path to feature parquet
TARGETS = ["accuracy", "fpos", "fmiss_extended"]
RANDOM_STATE = 42

In [4]:
# ---------------------------------------------------------------------------
# Load and clean data
# ---------------------------------------------------------------------------
loader = QCDataLoader()
df_raw = loader.load(PARQUET_PATH)
cleaner = QCDataCleaner()
df, clean_report = cleaner.clean(df_raw)

print(f"Total rows: {len(df):,}")
print(df["dataset_type"].value_counts().to_string())

Total rows: 33,193
dataset_type
hybrid    24211
paired     8982


In [5]:
# ---------------------------------------------------------------------------
# Hybrid → Paired transfer split
# ---------------------------------------------------------------------------
# Train: ALL hybrid rows (no filtering).
# Test:  paired rows that were matched to a GT unit only.
#        Unmatched paired rows (false_positive, unmatched, redundant, overmerged)
#        have no meaningful accuracy/fpos/fmiss and are excluded.
train_df = df[df["dataset_type"] == "hybrid"].copy()
test_df  = df[
    (df["dataset_type"] == "paired") & df["matched_gt_unit_id"].notna()
].copy()

print(f"Train (hybrid, all):          {len(train_df):,} rows")
print(f"Test  (paired, matched only): {len(test_df):,} rows")
print(f"\nPaired match_status breakdown:")
print(df[df["dataset_type"] == "paired"]["match_status"].value_counts().to_string())

Train (hybrid, all):          24,211 rows
Test  (paired, matched only): 207 rows

Paired match_status breakdown:
match_status
false_positive    5901
unmatched         2874
matched_best       135
well_detected       72


In [6]:
# ---------------------------------------------------------------------------
# Optional: per-recording z-score normalization
# ---------------------------------------------------------------------------
USE_NORMALIZER = True

if USE_NORMALIZER:
    normalizer = RecordingNormalizer()
    # Fit on ALL rows so every recording's stats are computed from its full
    # unit population (not just train or test rows).
    df_norm = normalizer.fit_transform(df)
    train_df = df_norm.loc[train_df.index].copy()
    test_df  = df_norm.loc[test_df.index].copy()
    z_cols = [c for c in train_df.columns if c.endswith("_z")]
    print(f"RecordingNormalizer applied — {len(z_cols)} _z columns added")

RecordingNormalizer applied — 13 _z columns added


In [7]:
# ---------------------------------------------------------------------------
# Build feature specs
# ---------------------------------------------------------------------------
builder = FeatureSetBuilder()
spec_unit_only = builder.build(train_df, mode="unit_only")

print(f"unit_only: {len(spec_unit_only.numeric_cols)} numeric, "
      f"{len(spec_unit_only.categorical_cols)} categorical")
wf_cols  = [c for c in spec_unit_only.numeric_cols if c.startswith("wf_bin_")]
acg_cols = [c for c in spec_unit_only.numeric_cols if c.startswith("acg_")]
print(f"  wf_bin_* columns: {len(wf_cols)}, acg_* columns: {len(acg_cols)}")

unit_only: 180 numeric, 0 categorical
  wf_bin_* columns: 30, acg_* columns: 40


In [8]:
# ---------------------------------------------------------------------------
# Run benchmark — dummy + lightgbm + neural (all targets)
# ---------------------------------------------------------------------------
trainer = NeuralTrainer(random_state=RANDOM_STATE)

print("Registered models:", trainer.registry.names())

model_names = [n for n in trainer.registry.names()
               if n in ("dummy", "lightgbm") or n.startswith("neural_")]
print("Running:", model_names)

Registered models: ['dummy', 'random_forest', 'lightgbm', 'neural_accuracy', 'neural_fpos', 'neural_fmiss_extended']
Running: ['dummy', 'lightgbm', 'neural_accuracy', 'neural_fpos', 'neural_fmiss_extended']


In [11]:
results = trainer.fit_all(
    train=train_df,
    test=test_df,
    targets=TARGETS,
    specs=[spec_unit_only],
    split_name="hybrid_to_paired",
    model_names=model_names,
)
print(f"Got {len(results)} TrainerResult objects")

KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# Results comparison table
# ---------------------------------------------------------------------------
rows = [
    {
        "target":       r.target,
        "model":        r.model_name,
        "n_train":      r.n_train,
        "n_test":       r.n_test,
        "mae":          round(r.mae,  4),
        "rmse":         round(r.rmse, 4),
        "r2":           round(r.r2,   4),
        "bias":         round(r.bias, 4),
    }
    for r in results
]

summary = pd.DataFrame(rows).sort_values(["target", "r2"], ascending=[True, False])
summary

,target,model,n_train,n_test,mae,rmse,r2,bias
0,accuracy,dummy,24211,207,0.2909,0.3270,-0.1869,-0.1298
1,accuracy,lightgbm,24211,207,0.2716,0.3410,-0.2910,-0.1838
4,fmiss_extended,dummy,24211,207,0.3238,0.3588,-0.5640,0.2155
5,fmiss_extended,lightgbm,24211,207,0.2811,0.3626,-0.5970,0.1947
2,fpos,dummy,24211,207,0.2128,0.2494,-0.0771,0.0667
3,fpos,lightgbm,24211,207,0.2579,0.3088,-0.6504,0.2014


In [ ]:
# ---------------------------------------------------------------------------
# Per-target highlight: best model per target
# ---------------------------------------------------------------------------
best = summary.loc[summary.groupby("target")["r2"].idxmax()]
print("Best model per target (R²):")
print(best[["target", "model", "r2", "mae"]].to_string(index=False))

Best model per target (R²):
        target model      r2    mae
      accuracy dummy -0.1869 0.2909
fmiss_extended dummy -0.5640 0.3238
          fpos dummy -0.0771 0.2128


In [ ]:
# ---------------------------------------------------------------------------
# Optional: scatter plot — true vs predicted (neural accuracy)
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

neural_acc = next(
    (r for r in results if r.model_name == "neural_accuracy" and r.target == "accuracy"),
    None,
)

if neural_acc is not None:
    p = neural_acc.predictions
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(p["true"], p["pred"], alpha=0.3, s=10)
    ax.plot([0, 1], [0, 1], "r--", lw=1)
    ax.set_xlabel("True accuracy")
    ax.set_ylabel("Predicted accuracy")
    ax.set_title(f"Neural CNN  R²={neural_acc.r2:.3f}  MAE={neural_acc.mae:.3f}")
    plt.tight_layout()
    plt.show()
else:
    print("neural_accuracy result not found — check model registration and torch availability")

neural_accuracy result not found — check model registration and torch availability
